# NB2. ROF monthly, annual, seasonal discharge at ocean outlets <a id='top'></a>

Use the following datasets 

1. reach-D19 gauge link ascii
2. D19 flow site geopackage
3. D19 discharge netCDF
4. monthly and yearly flow netCD (history file)

[1. Setupt](#setup)


[2. Loading discharge data](#load_discharge_data)

- Read monthly history files from archive. 
- Reference data: monthly discharge estimates at 922 big river mouths from Dai et al. 2019 data (D19)

[3. Read river, catchment, gauge information](#read_ancillary)

- catchment polygon (geopackage)
- gauge point (geopackage)
- gauge-catchment link (csv)
- outlet reach information (netCDF) including discharging ocean names

[4. Ocean discharge line plots](#922_rivers)

- total seasonal flow for oceans. 


In [ ]:
%matplotlib inline

import os, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask_jobqueue import PBSCluster
from dask.distributed import Client

from scripts.utility import load_yaml 
from scripts.utility import no_time_variable
from scripts.utility import read_shps
from scripts.utility import get_index_array

rivers_50m = cfeature.NaturalEarthFeature('physical', 'rivers_lake_centerlines', '50m')
land = cfeature.LAND

print("\nThe Python version: %s.%s.%s" % sys.version_info[:3])
print(xr.__name__, xr.__version__)
print(pd.__name__, pd.__version__)
print(gpd.__name__, gpd.__version__)

-------------------------
## 1. Analysis setup <a id='setup'></a>

**Please provide CESM case names and ROF grid name**

[go back to top](#top)

In [ ]:
# CESM case names and their runoff grid
plot_name = "test"

cases = {
    #'f09_f09_rHDMA':'rHDMA',
    #'f09_f09_rHDMAlk':'rHDMAlk',
    #'f09_f09_rHDMAlk_direct_to_outlet':'rHDMAlk',
    #'f09_f09_rHDMAlk_direct_to_outlet_no_flood':'rHDMAlk',
    #'f09_f09_rHDMAlk_mg17_irrig':'rHDMAlk_irrig',
    'test':'f09_f09',
    'f09_f09':'f09_f09',
    'mosart_f09_f09':'f09_f09_mosart',
}

parallel = True

figureSave = False

-------------------------
load config files and some parameters 

In [ ]:
setup = load_yaml("./setup/setup.yaml")

main_dir = setup["archive_dir"]  # CESM archive directory
domain_dir = setup[
    "ancillary_dir"
]  # ancillary directory including such as ROF domain, river network data
geospatial_dir = setup["geospatial_dir"]  # including shapefiles or geopackages
ref_flow_dir = setup["ref_flow_dir"]  # including observed or reference flow data

syr = setup["syr"]  # analysis start year
eyr = setup["eyr"]  # analysis end year

case_meta = setup["case_meta"]  # RO grid meta
catch_gpkg = setup["catch_gpkg"]  # catchment geopackage meta
reach_gpkg = setup["reach_gpkg"]  # reach geopackage meta
network_nc = setup["river_network"]  # river network meta

In [ ]:
oceans_list = [
    "arctic",
    "atlantic",
    "indian",
    "mediterranean",
    "pacific",
    "south_china",
    "global",
]
time_period = slice(f"{syr}-01-01", f"{eyr}-12-31")  # analysis time period
nyrs = eyr - syr + 1  # number of years
nmons = nyrs * 12  # number of months

-----
### dasks (optional)

In [ ]:
if parallel:
    cluster = PBSCluster(queue='casper', memory='10GB', processes=1)
    cluster.scale(jobs=10)
    client = Client(cluster)
    client

## 2. Loading discharge data <a id='load_discharge_data'></a>

[go back to top](#top)

### 2.1. Mmonthly/annual flow netCDFs
- month_data (xr dataset)
- year_data (xr dataset)
- seas_data (xr dataset)

In [ ]:
%%time

reachID = {}
month_data = {}
year_data = {}
seas_data = {}
for case, grid_name in cases.items():
    in_dire = os.path.join(main_dir, case, "rof/hist")
    model = case_meta[grid_name]["model"]
    domain = case_meta[grid_name]["domain_nc"]
    # monthly
    month_data[case] = (
        xr.open_mfdataset(
            f"{in_dire}/{case}.{model}.h0a.*.month.nc",
            data_vars="minimal",
            chunks={"time": 12},
        )
        .sel(time=time_period)
        .load()
    )
    # annual
    year_data[case] = (
        xr.open_mfdataset(
            f"{in_dire}/{case}.{model}.h0a.*.annual.nc",
            data_vars="minimal",
            chunks={"time": 1},
        )
        .sel(time=time_period)
        .load()
    )
    # seasonal (compute here instead of reading because of different time period)
    seas_data[case] = month_data[case].groupby("time.month").mean("time")
    vars_no_time = no_time_variable(month_data[case])
    seas_data[case][vars_no_time] = seas_data[case][vars_no_time].isel(
        month=0, drop=True
    )

    if domain == "None":
        reachID[case] = month_data[case]["reachID"].values
    else:
        reachID[case] = (
            xr.open_dataset(f"{domain_dir}/{domain}")["reachID"]
            .stack(seg=("lat", "lon"))
            .values
        )
    print(case)

### 2.2 D19 discharge data
- ds_q_obs_mon (xr datasets)
- ds_q_obs_yr (xr datasets)
- dr_q_obs_seasonal (xr datasets)

In [ ]:
%%time

# read monthly data
ds_q = xr.open_dataset(
    "%s/D09/coastal-stns-Vol-monthly.updated-May2019.mod.nc" % (ref_flow_dir),
    decode_times=False,
)
ds_q["time"] = xr.date_range(
    start="1900-01-01", end="2018-12-01", freq="MS", calendar="standard", use_cftime=True
)

# monthly
ds_q_obs_mon = ds_q[["FLOW", "lat", "lon"]].sel(time=time_period).set_coords(["lat","lon"])
# compute annual flow from monthly
ds_q_obs_yr = ds_q_obs_mon.resample(time="YE").mean(dim="time")
# compute annual cycle at monthly scale
dr_q_obs_seasonal = (
    ds_q_obs_mon.sel(time=time_period).groupby("time.month").mean("time")
)

## 3. Reading river, catchment, gauge infomation  <a id='read_meta'></a>

- catchment polygon (geopackage)
- gauge point (geopackage)
- gauge-catchment link (csv)
- outlet reach information (netCDF)

[go back to top](#top)

### 3.1. reach-D19 gauge link csv
- gauge_reach_lnk (dataframe)

In [ ]:
gauge_reach_lnk = {}
for case, grid_name in cases.items():
    gauge_reach_lnk[case] = pd.read_csv(
        "%s/D09/D09_925.%s.asc" % (ref_flow_dir, case_meta[grid_name]["network"])
    )

### 3.2 D19 flow site shapefile
- gauge_shp (dataframe)

In [ ]:
%%time

gauge_shp = gpd.read_file(
    os.path.join(ref_flow_dir, "D09", "geospatial", "D09_925.gpkg")
)
gauge_shp = gauge_shp[gauge_shp["id"] != 9999999]

In [ ]:
%%time

ocean_shp = gpd.read_file(os.path.join(geospatial_dir, "oceans.gpkg"))

### 3.3 Read river network information
- riv_ocean (dataframe)

In [ ]:
%%time

## read catchment geopackage
gdf_cat = {}
for case, grid_name in cases.items():
    network_name = case_meta[grid_name]["network"]
    cat_gpkg = os.path.join(
        geospatial_dir, catch_gpkg[network_name]["file_name"]
    )  # geopackage name
    id_name_cat = catch_gpkg[network_name]["id_name"]  # reach ID in geopackage
    var_list = [id_name_cat]
    if "lk" in grid_name:
        var_list.append("lake")
    gdf_cat[case] = read_shps([cat_gpkg], var_list)
    gdf_cat[case]['centroid_longitude'] = gdf_cat[case]['geometry'].centroid.x
    gdf_cat[case]['centroid_latitude'] = gdf_cat[case]['geometry'].centroid.y

In [ ]:
%%time

# read river outlet netcdf
riv_ocean = {}
for case, grid_name in cases.items():
    network = case_meta[grid_name]["network"]
    riv_ocean_file = os.path.join(
        domain_dir, network_nc[network]["file_name"].replace(".aug.nc", ".outlet.nc")
    )  # network netcdf name
    ds_rn_ocean = xr.open_dataset(riv_ocean_file).set_index(seg="seg_id")
    df_tmp = ds_rn_ocean.to_dataframe()
    riv_ocean[case] = pd.merge(
        gdf_cat[case], df_tmp, left_on=catch_gpkg[network]["id_name"], right_index=True
    )
    riv_ocean[case].rename(columns={catch_gpkg[network]["id_name"]:"hruid"}, inplace=True)

### 2.6 Merge gauge, outlet catchment dataframe

- gauge_shp1 (dataframe)

In [ ]:
%%time

# Merge gauge_reach lnk (dataframe) into gauge shapefile
gauge_shp1 = {}
for case, df in gauge_reach_lnk.items():
    network = case_meta[cases[case]]["network"]

    # df = df.loc[(df['flag'] == 0)]
    df1 = df.drop(columns=["riv_name"])
    df2 = pd.merge(gauge_shp, df1, how="inner", left_on="id", right_on="gauge_id")
    gauge_shp1[case] = pd.merge(
        df2,
        riv_ocean[case],
        how="inner",
        left_on="route_id",
        right_on=catch_gpkg[network]["id_name"],
    )

------
## 4. plot annual cycle for global oceans <a id='24_large_rivers'></a>

TODO: Referece flow plot should be independent from cases (network). Currently the last case plotted looks better matched with reference flow. 

[go back to top](#top)

In [ ]:
%time
# compute total discharge from only at the outlets in Dai/Trenberth data.
nrows = 4
ncols = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(7.25, 6.5))
plt.subplots_adjust(
    top=0.95, bottom=0.065, right=0.98, left=0.10, hspace=0.225, wspace=0.250
)  # create some space below the plots by increasing the bottom-value

for ix, ocean_name in enumerate(oceans_list):
    row = ix // 2
    col = ix % 2
    for case in cases:
        grid_name = cases[case]
        q_name = case_meta[grid_name]["flow_name"]
        color = case_meta[grid_name]["color"]

        if case_meta[grid_name]["network_type"] == "vector":
            if ocean_name == "global":
                id_list = gauge_shp1[case]["route_id"].values
            else:
                id_list = gauge_shp1[case][gauge_shp1[case]["ocean"] == ocean_name][
                    "route_id"
                ].values
            reach_index = get_index_array(reachID[case], id_list)
            dr_flow = seas_data[case][q_name].isel(seg=reach_index).sum(dim="seg")
            dr_flow.plot(ax=axes[row, col], linestyle="-", c=color, lw=0.75, label=case)

        elif case_meta[grid_name]["network_type"] == "grid":  # means 2d grid
            if ocean_name == "global":
                id_list = gauge_shp1[case]["route_id"].values
            else:
                id_list = gauge_shp1[case][gauge_shp1[case]["ocean"] == ocean_name][
                    "route_id"
                ].values

            reach_index = get_index_array(reachID[case], id_list)
            seas_data_vector = seas_data[case][q_name].stack(seg=("lat", "lon"))
            dr_flow = seas_data_vector.isel(seg=reach_index).sum(dim="seg")
            dr_flow.plot(ax=axes[row, col], linestyle="-", c=color, lw=0.75, label=case)

    # reference data
    if ocean_name == "global":
        id_list = gauge_shp1[case]["id"].values
    else:
        id_list = gauge_shp1[case][gauge_shp1[case]["ocean"] == ocean_name]["id"].values
    gauge_index = get_index_array(ds_q["id"].values, id_list)
    dr_obs = dr_q_obs_seasonal.isel(station=gauge_index).sum(dim="station")
    dr_obs["FLOW"].plot(
        ax=axes[row, col],
        linestyle="None",
        marker="o",
        markersize=2,
        c="k",
        label="D19",
    )

    axes[row, col].set_title("%d %s" % (ix + 1, ocean_name), fontsize=9)
    axes[row, col].set_xlabel("")
    if row < 3 and col==0 or row < 2 and col==1:
        axes[row, col].set_xticklabels("")
    if col == 0:
        axes[row, col].set_ylabel("Mon. flow [m$^3$/s]", fontsize=9)
    else:
        axes[row, col].set_ylabel("")
    axes[row, col].tick_params("both", labelsize="x-small")

# Legend- make space below the plot-raise bottom. there will be an label below the second last (bottom middle) ax, thanks to the bbox_to_anchor=(x, y) with a negative y-value.
axes[row, col].legend(
    loc="center left", bbox_to_anchor=(1.10, 0.40, 0.75, 0.1), ncol=1, fontsize="small"
)

for jx in range(ix + 1, nrows * ncols):
    row = jx // 2
    col = jx % 2
    fig.delaxes(axes[row][col])

if figureSave:
    plt.savefig(f"./NB2_Fig1_ocean_discharge_season_{plot_name}_gauge_only.png", dpi=200)

In [ ]:
%time
# compute zonal mean total discharge from all the outlets in the modeled river networks.
ocean_name='global' # atlantic, pacific, indian, or global

nrows = 1
ncols = 1
fig, axes = plt.subplots(nrows, ncols, figsize=(6.25, 5.0))
plt.subplots_adjust(
    top=0.95, bottom=0.065, right=0.98, left=0.10, hspace=0.225, wspace=0.250
)  # create some space below the plots by increasing the bottom-value

axes_right = axes.twinx()

# Define latitude bins every 1 degree
lat_bins = np.arange(-80, 81, 1.0)
# Compute bin centers for labeling
lat_bin_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])

for case in cases:
    print(case)
    grid_name = cases[case]
    q_name = case_meta[grid_name]["flow_name"]
    color = case_meta[grid_name]["color"]

    # extract flow data and latitude at outlet locations (index)
    if ocean_name == "global":
        id_list = gauge_shp1[case]["route_id"].values
    else:
        id_list = gauge_shp1[case][gauge_shp1[case]["ocean"] == ocean_name][
            "route_id"
        ].values
    reach_index = get_index_array(reachID[case], id_list)

    if case_meta[grid_name]["network_type"] == "vector":
        # 1. Extract the mapping from riv_ocean (HRU → latitude)
        lat_map = riv_ocean[case].set_index("hruid")["centroid_latitude"]
        
        # 2. Convert that Series into an xarray DataArray
        lat_da = xr.DataArray(
            lat_map,
            dims=["hruid"],
            coords={"hruid": lat_map.index},
            name="lat"
        )
        
        # 3. Align `lat_da` with the reachID coordinate in seas_data
        #    (We assume reachID corresponds to hruid values)
        lat_for_reach = lat_da.sel(hruid=xr.DataArray(seas_data[case]["reachID"].values, dims=["seg"]))
        
        # 4. Assign it as a new variable in seas_data
        seas_data[case]['lat'] = lat_for_reach
        
        ds_flow = seas_data[case][[q_name, 'lat']].isel(seg=reach_index)

        # Group by latitude bins and take the mean across segments in each bin
        zonal_mean = (
            ds_flow[q_name].mean(dim='month')
            .groupby_bins(ds_flow["lat"], bins=lat_bins, labels=lat_bin_centers)
            .sum(dim="seg", skipna=True)
        )/ 1000000

    elif case_meta[grid_name]["network_type"] == "grid":  # means 2d grid
        
        seas_data_vector = seas_data[case][q_name].stack(seg=("lat", "lon"))
        dr_flow = seas_data_vector.isel(seg=reach_index)
        zonal_mean = (
            dr_flow.mean(dim='month')
            .groupby_bins(dr_flow["lat"], bins=lat_bins, labels=lat_bin_centers)
            .sum(dim="seg", skipna=True)
        ) / 1000000
 
    # Rename the bin dimension to 'lat' for clarity
    zonal_mean = zonal_mean.rename({"lat_bins": "lat"})
    zonal_mean = zonal_mean.assign_coords(lat=("lat", lat_bin_centers))

    # plot zonal mean discharge per 1-degree lat
    zonal_mean.plot(ax=axes, linestyle="-", c=color, lw=0.75, label=case)
    # plot cumulative zonal mean discharge per 1-degree lat
    zonal_mean.cumsum(dim='lat').plot(ax=axes_right, linestyle="--", c=color, lw=1.0, label=case)
 
# reference data
if ocean_name == "global":
    id_list = gauge_shp1[case]["id"].values
else:
    id_list = gauge_shp1[case][gauge_shp1[case]["ocean"] == ocean_name]["id"].values
gauge_index = get_index_array(ds_q["id"].values, id_list)
dr_obs = dr_q_obs_seasonal.isel(station=gauge_index)
zonal_mean_obs = (
            dr_obs["FLOW"].mean(dim='month')
            .groupby_bins(dr_obs["lat"], bins=lat_bins, labels=lat_bin_centers)
            .sum(dim="station", skipna=True)
        ) / 1000000

zonal_mean_obs.plot(
    ax=axes,
    linestyle="-",
    #marker="o",
    #markersize=2,
    c="k",
    label="D19",zorder=0)

axes.set_title("%s" % (ocean_name), fontsize=10)
axes.set_xlabel("Latitude")

axes.set_xticks(np.arange(-80,81,20))
axes.set_xticklabels(["80S","60S","40S","20S","0","20N","40N","60N","80N"])
axes.set_ylabel("Total annual discharge into ocean [10$^6$ m$^3$/s per degree lat]", fontsize=10)
axes.tick_params("both", labelsize="small")

axes_right.set_ylabel("Total annual discharge accumulated from 90S [10$^6$ m$^3$/s]", fontsize=10)
axes_right.tick_params("both", labelsize="small")

# Legend- make space below the plot-raise bottom. there will be an label below the second last (bottom middle) ax, thanks to the bbox_to_anchor=(x, y) with a negative y-value.
axes.legend(
    loc="center left", bbox_to_anchor=(0.025, 0.875, 0.75, 0.1), ncol=1, fontsize="small"
)

if figureSave:
    plt.savefig(f"./NB2_Fig3_global_zonal_mean_annal_discharge_{plot_name}_gauge_only.png", dpi=300)

In [ ]:
%time
# compute total discharge from all the outlets in the modeled river networks.
nrows = 4
ncols = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(7.25, 6.5))
plt.subplots_adjust(
    top=0.95, bottom=0.065, right=0.98, left=0.10, hspace=0.225, wspace=0.250
)  # create some space below the plots by increasing the bottom-value

for ix, ocean_name in enumerate(oceans_list):
    row = ix // 2
    col = ix % 2
    for case in cases:
        grid_name = cases[case]
        
        q_name = case_meta[grid_name]["flow_name"]
        color = case_meta[grid_name]["color"]
        idname = catch_gpkg[case_meta[grid_name]['network']]['id_name']
        
        if case_meta[grid_name]["network_type"] == "vector":
            if ocean_name == "global":
                id_list = riv_ocean[case][(riv_ocean[case]['outletType']==1)][idname].values #(riv_ocean[case]['ocean']!='land') & 
            else:
                id_list = riv_ocean[case][(riv_ocean[case]['ocean']==ocean_name) & (riv_ocean[case]['outletType']==1)][idname].values 
            reach_index = get_index_array(reachID[case], id_list)
            dr_flow = seas_data[case][q_name].isel(seg=reach_index).sum(dim="seg")
            dr_flow.plot(ax=axes[row, col], linestyle="-", c=color, lw=0.75, label=case)

        elif case_meta[grid_name]["network_type"] == "grid":  # means 2d grid
            if ocean_name == "global":
                id_list = riv_ocean[case][(riv_ocean[case]['outletType']==1)][idname].values #(riv_ocean[case]['ocean']!='land') & 
            else:
                id_list = riv_ocean[case][(riv_ocean[case]['ocean']==ocean_name) & (riv_ocean[case]['outletType']==1)][idname].values 

            reach_index = get_index_array(reachID[case], id_list)
            seas_data_vector = seas_data[case]['TOTAL_DISCHARGE_TO_OCEAN_LIQ'].stack(seg=("lat", "lon"))
            dr_flow = seas_data_vector.isel(seg=reach_index).sum(dim="seg")
            dr_flow.plot(ax=axes[row, col], linestyle="-", c=color, lw=0.75, label=case)

    axes[row, col].set_title("%d %s" % (ix + 1, ocean_name), fontsize=9)
    axes[row, col].set_xlabel("")
    if row < 3 and col==0 or row < 2 and col==1:
        axes[row, col].set_xticklabels("")
    if col == 0:
        axes[row, col].set_ylabel("Mon. flow [m$^3$/s]", fontsize=9)
    else:
        axes[row, col].set_ylabel("")
    axes[row, col].tick_params("both", labelsize="x-small")

# Legend- make space below the plot-raise bottom. there will be an label below the second last (bottom middle) ax, thanks to the bbox_to_anchor=(x, y) with a negative y-value.
axes[row, col].legend(
    loc="center left", bbox_to_anchor=(1.10, 0.40, 0.75, 0.1), ncol=1, fontsize="small"
)

for jx in range(ix + 1, nrows * ncols):
    row = jx // 2
    col = jx % 2
    fig.delaxes(axes[row][col])

if figureSave:
    plt.savefig(f"./NB2_Fig1_ocean_discharge_season_{plot_name}.png", dpi=200)

In [ ]:
%time
# compute zonal mean total discharge from all the outlets in the modeled river networks.
ocean_name='global'

nrows = 1
ncols = 1
fig, axes = plt.subplots(nrows, ncols, figsize=(6.5, 5.0))
plt.subplots_adjust(
    top=0.95, bottom=0.065, right=0.98, left=0.10, hspace=0.225, wspace=0.250
)  # create some space below the plots by increasing the bottom-value

axes_right = axes.twinx()

# Define latitude bins every 1 degree
lat_bins = np.arange(-80, 81, 1.0)
# Compute bin centers for labeling
lat_bin_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])

for case in cases:
    print(case)
    grid_name = cases[case]
    q_name = case_meta[grid_name]["flow_name"]
    color = case_meta[grid_name]["color"]
    idname = "hruid"#catch_gpkg[case_meta[grid_name]['network']]['id_name']

    # 5. extract flow data and latitude at outlet locations (index)
    if ocean_name == "global":
        id_list = riv_ocean[case][(riv_ocean[case]['ocean']!='land') & (riv_ocean[case]['outletType']==1)][idname].values
    else:
        id_list = riv_ocean[case][(riv_ocean[case]['ocean']==ocean_name) & (riv_ocean[case]['outletType']==1)][idname].values
    reach_index = get_index_array(reachID[case], id_list)

    if case_meta[grid_name]["network_type"] == "vector":
        # 1. Extract the mapping from riv_ocean (HRU → latitude)
        lat_map = riv_ocean[case].set_index("hruid")["centroid_latitude"]
        
        # 2. Convert that Series into an xarray DataArray
        lat_da = xr.DataArray(
            lat_map,
            dims=["hruid"],
            coords={"hruid": lat_map.index},
            name="lat"
        )
        
        # 3. Align `lat_da` with the reachID coordinate in seas_data
        #    (We assume reachID corresponds to hruid values)
        lat_for_reach = lat_da.sel(hruid=xr.DataArray(seas_data[case]["reachID"].values, dims=["seg"]))
        
        # 4. Assign it as a new variable in seas_data
        seas_data[case]['lat'] = lat_for_reach
        
        ds_flow = seas_data[case][[q_name, 'lat']].isel(seg=reach_index)

        # Group by latitude bins and take the mean across segments in each bin
        zonal_mean = (
            ds_flow[q_name].mean(dim='month')
            .groupby_bins(ds_flow["lat"], bins=lat_bins, labels=lat_bin_centers)
            .sum(dim="seg", skipna=True)
        )

    elif case_meta[grid_name]["network_type"] == "grid":  # means 2d grid
        
        seas_data_vector = seas_data[case]['TOTAL_DISCHARGE_TO_OCEAN_LIQ'].stack(seg=("lat", "lon"))
        dr_flow = seas_data_vector.isel(seg=reach_index)
        zonal_mean = (
            dr_flow.mean(dim='month')
            .groupby_bins(dr_flow["lat"], bins=lat_bins, labels=lat_bin_centers)
            .sum(dim="seg", skipna=True)
        )
 
    # Rename the bin dimension to 'lat' for clarity
    zonal_mean = zonal_mean.rename({"lat_bins": "lat"})
    zonal_mean = zonal_mean.assign_coords(lat=("lat", lat_bin_centers))

    # plot zonal mean discharge per 1-degree lat
    zonal_mean.plot(ax=axes, linestyle="-", c=color, lw=0.75, label=case)
    # plot cumulative zonal mean discharge per 1-degree lat
    zonal_mean.cumsum(dim='lat').plot(ax=axes_right, linestyle="--", c=color, lw=1.0, label=case)

axes.set_title("%s" % (ocean_name), fontsize=10)
axes.set_xlabel("Latitude")
axes.set_xticks(np.arange(-80,81,20))
axes.set_xticklabels(["80S","60S","40S","20S","0","20N","40N","60N","80N"])
axes.set_ylabel("Total annual discharge into ocean [m$^3$/s per degree lat]", fontsize=10)
axes.tick_params("both", labelsize="small")

axes_right.set_ylabel("Total annual discharge accumulated from 90S [m$^3$/s]", fontsize=10)
axes_right.tick_params("both", labelsize="small")

# Legend- make space below the plot-raise bottom. there will be an label below the second last (bottom middle) ax, thanks to the bbox_to_anchor=(x, y) with a negative y-value.
axes.legend(
    loc="center left", bbox_to_anchor=(0.025, 0.875, 0.75, 0.1), ncol=1, fontsize="small"
)

if figureSave:
    plt.savefig(f"./NB2_Fig3_global_zonal_mean_annal_discharge_{plot_name}.png", dpi=200)

In [ ]:
seas_data_vector = seas_data[case][['TOTAL_DISCHARGE_TO_OCEAN_LIQ','RIVER_DISCHARGE_OVER_LAND_LIQ']].stack(seg=("lat", "lon"))

In [ ]:
outlet_indices =np.argwhere(seas_data_vector['TOTAL_DISCHARGE_TO_OCEAN_LIQ'].isel(month=6).values>0)

In [ ]:
ix = get_index_array(reachID['f09_f09_mg17_mosart'], [120625])
print(seas_data_vector['seg'].isel(seg=ix).values)

ixup = get_index_array(reachID['f09_f09_mg17_mosart'], [120626])
ixup_miz = get_index_array(reachID['f09_f09_mg17'], [120626])
print(seas_data_vector['seg'].isel(seg=ixup).values)

ixup1 = get_index_array(reachID['f09_f09_mg17_mosart'], [121347])
print(seas_data_vector['seg'].isel(seg=ixup1).values)


In [ ]:
seas_data_vector['TOTAL_DISCHARGE_TO_OCEAN_LIQ'].isel(seg=ix).plot(label='total_discharge_to ocean')
seas_data_vector['RIVER_DISCHARGE_OVER_LAND_LIQ'].isel(seg=ixup).plot(label='river_discharge_one_upstream')
seas_data_vector['RIVER_DISCHARGE_OVER_LAND_LIQ'].isel(seg=ixup1).plot(label='river_discharge_one_upstream1')

#seas_data['f09_f09_mg17']['DWroutedRunoff'].isel(seg=ixup_miz).plot(ls='--')
plt.legend()

In [ ]:
print(seas_data['f09_f09_mg17']['DWroutedRunoff'].isel(seg=ixup_miz).sum().values)
print(seas_data_vector['RIVER_DISCHARGE_OVER_LAND_LIQ'].isel(seg=ixup).sum().values)